In [ ]:
import re
import glob
import pandas as pd
import matplotlib.pyplot as plt

def read_log(log_path):
    training_data = {} # To store metric values
    test_data = {} # To store test result summary

    train_time = 0.0 # To store accumulated training time (in seconds)
    eval_time = 0.0 # To store accumulated eval time (in seconds)

    trainable_params = None  # To store "Trainable parameters: xxxxxxxx"
    flops = None             # To store "FLOPs: xxxxxx.xx"

    with open(log_path, "r") as f:
        for line in f:
            # Extracting per-epoch metrics (mrr@10, recall@10, precision@10, etc.)
            metric_pattern = r"(\w+@\d+)\s*:\s*([\d.]+)"
            matches = re.findall(metric_pattern, line)
            for metric, value in matches:
                if metric not in training_data:
                    training_data[metric] = []
                training_data[metric].append(float(value))

            # Extract per-epoch Training Loss. 
            # Example: epoch 0 training [time: 282.57s, train loss: 325.2666]
            m_loss = re.search(r"train loss:\s*([\d.]+)", line)
            if m_loss:
                if 'loss' not in training_data:
                    training_data["loss"] = []
                training_data['loss'].append(float(m_loss.group(1)))

            # Extract Trainable Parameters. 
            # Example: Trainable parameters: 168128
            params_match = re.search(r"Trainable parameters:\s*([\d]+)", line)
            if params_match:
                trainable_params = int(params_match.group(1))

            # Extract FLOPs. 
            # Example: Tue 18 Nov 2025 23:07:57 INFO  FLOPs: 128.0
            flops_match = re.search(r"FLOPs:\s*([\d.]+)", line)
            if flops_match:
                flops = float(flops_match.group(1))

            # Accumulate training and evaluation time from epoch logs
            m_train = re.search(r"epoch\s+(\d+)\s+training\s+\[time:\s*([\d.]+)s", line)
            if m_train:
                train_time += float(m_train.group(2))

            m_eval = re.search(r"epoch\s+(\d+)\s+evaluating\s+\[time:\s*([\d.]+)s", line)
            if m_eval:
                eval_time += float(m_eval.group(2))

            # Match summary lines containing test results
            if "INFO  test result:" in line:
                test_data = {m[0]: float(m[1] or m[2]) for m in re.findall(r"\('([^']+)', (?:np\.float64\(([\d.]+)\)|([\d.]+))\)", line)}


    epochs = len(next(iter(training_data.values())))  # Get the number of epochs

    return training_data, test_data, train_time, eval_time, epochs, trainable_params, flops

def normalize_loss(loss):
    loss = np.array(loss)
    return 1 - (loss - loss.min()) / (loss.max() - loss.min())

def plot(model, dataset, datatype, training_data, epochs, save_path):
    # Define colors for each metric (up to 9 different)
    colors = ['tab:red', 'tab:green', 'tab:blue', 'tab:purple', 'tab:orange', 'tab:brown', 'tab:pink', 'tab:cyan', 'tab:gray']
    
    p_metrics = ['loss', 'ndcg@10', 'hit@10']

    X = range(epochs)

    # Plot
    plt.figure(figsize=(10, 6))
    for c, m in enumerate(p_metrics):
        if m == 'loss':
            max_val = max(training_data[m])
            if max_val != 0:
                plt.plot(X, [val/max_val for val in training_data[m]], marker='.', linestyle=':', color=colors[c % len(colors)], 
                     label= r'$\frac{\text{loss}}{\text{max loss}}$' + f', where max loss = {max_val}')
        else:
            plt.plot(X, training_data[m], marker='.', linestyle=':', color=colors[c % len(colors)],
                     label= f'{m}')
            max_x = max(X, key=lambda i: training_data[m][i])
            plt.plot(max_x, training_data[m][max_x], marker='o', color=colors[c % len(colors)], 
                     markersize=10, markerfacecolor='none', markeredgewidth=1)

    plt.title(f"Validation metrics over epochs for:\n" + 
            f"{model} - {dataset} - {datatype}")
    plt.xlabel("Epoch")
    plt.ylabel("Metric Value")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.ylim(0, 1)
    plt.savefig(save_path+f'{model}-{dataset}-{datatype}', bbox_inches='tight')
    #plt.show()


models = ['ItemKNN', 'BPR', 'NeuMF', 'GCMC']
datasets = ['ml-100k', 'jester', 'amazon_all_beauty', 'douban', 'netflix']
datatypes = ['expR', 'impR', 'impB']  
MovieLens, Jester, Amazon, Douban, Netflix = range(5)
ItemKNN, BPR, NeuMF, GCMC = range(4)
expR, impR, impB = range(3)

metrics = ['recall@10', 'precision@10', 'mrr@10', 'ndcg@10', 'hit@10'] + \
            ['recall@20', 'precision@20', 'mrr@20', 'ndcg@20', 'hit@20']
 
columns = ['model', 'dataset', 'datatype'] + metrics + ['epochs', 'avg train time', 'avg eval time', 'FLOPs', 'trainable params']
df = pd.DataFrame(columns=columns)

read_path = 'data/log/'
save_path = 'data/result/'

for t in [expR, impR, impB]:
    for d in [Amazon, MovieLens ,Douban, Jester]:
        for m in [ItemKNN, BPR, NeuMF, GCMC]:
            model = models[m]
            dataset = datasets[d]
            datatype = datatypes[t]
            # pattern: model--dataset--datatype-???.log
            log_read_pattern = read_path+f'{model}-{dataset}--{datatype}-*.log'
            for log in glob.glob(log_read_pattern):
                training_data, test_data, train_time, eval_time, epochs, trainable_params, flops = read_log(log)

                # --- Test scores ---
                row = {
                    'model': model,
                    'dataset': dataset,
                    'datatype': datatype,
                    'epochs': epochs,
                    'avg train time': round(train_time / epochs, 2),
                    'avg eval time': round(eval_time / epochs, 2),
                    'trainable_params': trainable_params, 
                    'FLOPs': flops,
                    'trainable params': trainable_params,
                }
                # fill metrics (NaN if missing)
                for x in metrics:
                    row[x] = test_data.get(x, float('NaN'))

                df.loc[len(df)] = row  # append row

                plot(model, dataset, datatype, training_data, epochs, save_path)

#print(df.head())
df.to_csv(save_path+'df.csv', sep='\t', index=False)

In [ ]:
import pandas as pd

# =========================
# CONFIGURATION
# =========================
CSV_PATH = "data/result/df.csv"
OUTPUT_TEX = "data/result/df_table.tex"
CSV_SEPARATOR = "\t"

ID_COLUMNS = [
    "model",
    "dataset",
    "datatype",
]
EFFECTIVENESS_10_COLUMNS = [
    "recall@10",
    "precision@10",
    "mrr@10",
    "ndcg@10",
    "hit@10",
]
EFFECTIVENESS_20_COLUMNS = [
    "recall@20",
    "precision@20",
    "mrr@20",
    "ndcg@20",
    "hit@20",
]
EFFICIENCY_COLUMNS  = [
    "epochs",
    "avg train time",
    "avg eval time",
    "FLOPs",
    "trainable params",
]

COLUMNS_TO_USE = ID_COLUMNS + EFFECTIVENESS_10_COLUMNS

# Formatting
FLOAT_FMT = "{:.4f}"
BOLD_TIES = True
EPS = 1e-12

# Row rules (data rows only)
DOTTED_EVERY = 4
DASHED_EVERY = 16

# =========================
# HELPERS
# =========================
def escape_underscores(s: str) -> str:
    # Escape "_" for LaTeX. (Keep it simple: just what you asked for.)
    return s.replace("_", r"\_")

# =========================
# LOAD DATA
# =========================
df = pd.read_csv(CSV_PATH, sep=CSV_SEPARATOR)
df = df[COLUMNS_TO_USE].copy()

metric_cols = [c for c in df.columns if c not in ID_COLUMNS]

# Make metrics numeric
for c in metric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Escape underscores in ID columns (string content)
for c in ID_COLUMNS:
    df[c] = df[c].astype(str).map(escape_underscores)

# Escape underscores in headers too
escaped_headers = [escape_underscores(c) for c in df.columns]

# =========================
# HIGHLIGHT BEST PER METRIC COLUMN
# =========================
bold_group_keys = ["dataset", "datatype"]
underline_group_keys = ["model", "dataset"]

for c in metric_cols:
    col = df[c]

    # Best per (dataset, datatype)
    best_group = df.groupby(bold_group_keys)[c].transform("max")

    # Best datatype per (model, dataset)
    best_datatype = df.groupby(underline_group_keys)[c].transform("max")

    formatted = []

    for x, bg, bd in zip(col.tolist(), best_group.tolist(), best_datatype.tolist()):
        if pd.isna(x):
            formatted.append("")
            continue

        val = float(x)
        s = FLOAT_FMT.format(val)

        is_bold = not pd.isna(bg) and abs(val - bg) <= EPS
        is_underline = not pd.isna(bd) and abs(val - bd) <= EPS

        if is_bold and is_underline:
            formatted.append(f"\\underline{{\\textbf{{{s}}}}}")
        elif is_bold:
            formatted.append(f"\\textbf{{{s}}}")
        elif is_underline:
            formatted.append(f"\\underline{{{s}}}")
        else:
            formatted.append(s)

    df[c] = formatted


# =========================
# BUILD TABULARRAY (tblr) LATEX
# =========================
n_cols = len(df.columns)
n_data = len(df)

# column spec: left for ID cols, right for metrics
colspec = ("l" * len(ID_COLUMNS)) + ("r" * (n_cols - len(ID_COLUMNS)))

# IMPORTANT: In tabularray, hline{1} is the top border,
# hline{2} is the rule after row 1, etc.
# Header is row 1. Data rows are 2..(n_data+1).
# A rule "after data row k" (k starting at 1) is after overall row (k+1),
# i.e. hline{(k+1)+1} = hline{k+2}.
dotted_hlines = []
dashed_hlines = []

for k in range(1, n_data + 1):  # k = data row index (1-based)
    hline_index = k + 2         # rule after data row k
    if k % DASHED_EVERY == 0:
        dashed_hlines.append(hline_index)
    elif k % DOTTED_EVERY == 0:
        dotted_hlines.append(hline_index)

def join_ints(xs):
    return ",".join(str(x) for x in xs)

lines_opts = []
if dotted_hlines:
    #lines_opts.append(f"hline{{{join_ints(dotted_hlines)}}} = {{solid}}")
    lines_opts.append(f"hline{{{join_ints(dotted_hlines)}}} = {{dashed}}")
if dashed_hlines:
    idxs = join_ints(dashed_hlines)
    lines_opts.append(f"hline{{{idxs}}} = {{1}}{{-}}{{solid}}")
    lines_opts.append(f"hline{{{idxs}}} = {{2}}{{-}}{{solid}}")
    #lines_opts.append(f"hline{{{join_ints(dashed_hlines)}}} = {{solid}}")

tblr_opts = ",\n  ".join([
    f"colspec={{ {colspec} }}",
    "row{1}={font=\\bfseries\\large}",                 # bigger header
    f"cell{{1}}{{1-{n_cols}}}={{halign=l}}",           # header left-align
    f"cell{{2-Z}}{{1-{len(ID_COLUMNS)}}}={{halign=l}}",# ID cells left-align
    f"cell{{2-Z}}{{{len(ID_COLUMNS)+1}-{n_cols}}}={{halign=r}}",  # metric cells right-align

    # Full-width solid rules: top of header, bottom of header, bottom of table
    "hline{1} = {1}{-}{solid}", # top rule
    "hline{1} = {2}{-}{solid}", # top rule
    #"hline{1} = {3}{-}{solid}", # top rule
    "hline{2} = {1}{-}{solid}", # rule under header
    "hline{2} = {2}{-}{solid}", # rule under header
    #"hline{2} = {3}{-}{solid}", # rule under header
    *lines_opts,
    "hline{Z} = {1}{-}{solid}", # bottom rule
    "hline{Z} = {2}{-}{solid}", # bottom rule
    #"hline{Z} = {3}{-}{solid}", # bottom rule
    #"hline{Z}={solid}",   
])

header_row = " & ".join(escaped_headers) + r" \\"

body_rows = []
for _, row in df.iterrows():
    body_rows.append(" & ".join(row.tolist()) + r" \\")

latex_lines = []
latex_lines.append("% Auto-generated by Python (pandas) + tabularray")
latex_lines.append("% Requires in LaTeX preamble:")
latex_lines.append("% \\usepackage{tabularray}")
latex_lines.append("% \\UseTblrLibrary{booktabs}")
latex_lines.append("")
latex_lines.append("\\begin{tblr}[]{")
latex_lines.append("  " + tblr_opts)
latex_lines.append("}")
latex_lines.append(header_row)
latex_lines.extend(body_rows)
latex_lines.append("\\end{tblr}")
latex_lines.append("")

with open(OUTPUT_TEX, "w", encoding="utf-8") as f:
    f.write("\n".join(latex_lines))

print(f"Saved LaTeX table to {OUTPUT_TEX}")

In [ ]:
import pandas as pd
from IPython.display import HTML, display

df = pd.read_csv("data/result/df.csv", sep="\t")  # use sep="," if needed

rows = []
rows.append("<table style='border-collapse:collapse; width:100%; font-family:system-ui;'>")
# header
rows.append("<thead><tr>" + "".join(
    f"<th style='text-align:left; padding:6px; border-bottom:2px solid #000; font-weight:700;'>{c}</th>"
    for c in df.columns
) + "</tr></thead><tbody>")

# body
for i, r in enumerate(df.itertuples(index=False), start=1):
    rows.append("<tr>" + "".join(
        f"<td style='padding:6px; vertical-align:top;'>{'' if pd.isna(v) else v}</td>"
        for v in r
    ) + "</tr>")

    if i != len(df):
        if i % 12 == 0:
            rows.append(f"<tr><td colspan='{len(df.columns)}' style='border-top:2px dotted #000;'></td></tr>")
        elif i % 3 == 0:
            rows.append(f"<tr><td colspan='{len(df.columns)}' style='border-top:1px dotted #000;'></td></tr>")

rows.append("</tbody></table>")
display(HTML("".join(rows)))